# LLMs na GPU do Kaggle: ordem de leitura em Diários Oficiais

Roda LLMs abertas recentes nas **mesmas páginas e com o mesmo protocolo** dos outros métodos
(blocos embaralhados, mesma referência, mesmas métricas), na **segmentação v3**, no teste (89
páginas) e no `golden_v3` (47 páginas anotadas à mão), e registra a latência por página.

Modelos (biblioteca do Ollama, setembro de 2026), todos com o raciocínio desligado, para a
comparação ser com a mesma saída direta: **Qwen 3.8 27B** (18 GB), **Gemma 4 31B** (20 GB) e
**Qwen 3.6 35B** (23 GB, mistura de especialistas, a mais rápida). Cada um é dividido entre as
duas T4. Se as 12 horas não bastarem, a execução não começa o que não cabe (`LIMITE_HORAS`) e o
que já terminou fica no zip; rode de novo só com os modelos que faltaram.

Rode **depois** de `kaggle_treino_ordem.ipynb`: a tabela final fica completa se as predições
daquele notebook já estiverem no projeto.

**Antes de rodar**
1. *Settings* → *Accelerator*: **GPU T4 x2** · *Internet*: **On** (exige conta verificada por telefone).
2. *Add Input* → o dataset com `tcc_ordem_leitura.zip`.
3. Ajuste a célula de configuração logo abaixo.
4. *Save Version* → **Save & Run All (Commit)**: roda em segundo plano, por até 12 horas,
   mesmo com o navegador fechado.
5. No fim, baixe `resultados_kaggle.zip` na aba *Output* e, no projeto, rode
   `venv\Scripts\python.exe src\kaggle\importar_resultados.py resultados_kaggle.zip`.

In [ ]:
# ===== CONFIGURACAO =====
MODELOS = ["qwen3.8:27b", "gemma4:31b", "qwen3.6:35b"]  # rodam nesta ordem, divididos entre as duas T4
FEW_SHOT = [0, 2]        # 0 = zero-shot; 2 = dois exemplos resolvidos (paginas de TREINO) no prompt
PARTICOES = ["teste", "golden:_avaliacao/golden_v3"]  # teste = 89 paginas do protocolo; golden_v3 = 47 anotadas
LIMITE_HORAS = 10.5      # nao comeca uma rodada nova depois disso (a sessao do Kaggle cai em 12 h)
MAX_BLOCOS = 200         # 200 = todas as paginas
APAGAR_MODELO_APOS_USO = True   # libera o disco antes de baixar o proximo modelo
# Modelos com raciocinio (qwen3, deepseek-r1) rodam com o raciocinio desligado (--sem-pensar).

In [ ]:
import os, sys, glob, json, time, shutil, subprocess, urllib.request

achados = glob.glob("/kaggle/input/**/data/processed/splits.json", recursive=True)
assert achados, "Pacote nao encontrado: use Add Input e adicione o dataset com tcc_ordem_leitura.zip"
origem = os.path.dirname(os.path.dirname(os.path.dirname(achados[0])))
PROJ = "/kaggle/working/tcc"
if not os.path.exists(PROJ):
    shutil.copytree(origem, PROJ)
os.chdir(PROJ)
os.makedirs("models/layoutlm_ordem", exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)

def rodar(cmd, mostrar=True, ultimas_n=3):
    if isinstance(cmd, str):
        cmd = cmd.split()
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    ultimas = []
    for linha in proc.stdout:
        if mostrar:
            print(linha, end="", flush=True)
        ultimas = (ultimas + [linha])[-ultimas_n:]
    codigo = proc.wait()
    if not mostrar:
        print("".join(ultimas).strip()[-500:])
    print(f"  [{time.time() - t0:.0f} s, codigo {codigo}]", flush=True)
    return codigo

def zipar():
    """Empacota resultados/ em /kaggle/working/resultados_kaggle.zip. Chamado ao fim e depois de cada
    etapa longa: se a sessao estourar as 12 h, o que ja terminou nao se perde."""
    import zipfile
    destino = "/kaggle/working/resultados_kaggle.zip"
    with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as z:
        for arq in glob.glob("resultados/**/*", recursive=True):
            if os.path.isfile(arq) and not arq.endswith(".pt"):
                z.write(arq)
        for arq in glob.glob("models/layoutlm_ordem/*_config.json"):
            z.write(arq)
        for arq in glob.glob("docs/figuras/*.png"):
            z.write(arq)
        for arq in glob.glob("/kaggle/working/*.log"):
            z.write(arq, "logs/" + os.path.basename(arq))
    print("Pronto:", destino, f"({os.path.getsize(destino) / 2**20:.1f} MB). Baixe na aba Output.")

INICIO_SESSAO = time.time()
print("Projeto em", PROJ)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Instala e liga o Ollama (o mesmo servidor de LLM usado no notebook local)
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y pciutils lshw zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /kaggle/working/instalacao_ollama.log 2>&1; tail -n 2 /kaggle/working/instalacao_ollama.log

os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
servidor = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                            stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        versao = json.loads(urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2).read())
        print("Ollama", versao["version"], "no ar")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("O Ollama nao subiu; veja /kaggle/working/ollama.log")

In [ ]:
# Latencia de ingenua, XY-cut, BiLSTM e LayoutLM NESTA maquina, com o mesmo roteiro de medicao
rodar([sys.executable, "src/eval/latencia_hardware.py", "--maquina", "kaggle_t4"])

In [ ]:
RACIOCINIO = ("qwen3", "deepseek-r1", "magistral", "gemma4", "gpt-oss")   # qwen3 cobre qwen3.6 e qwen3.8
horas = lambda: (time.time() - INICIO_SESSAO) / 3600
for modelo in MODELOS:
    if horas() > LIMITE_HORAS:
        print(f"[!] {horas():.1f} h de sessao: {modelo} fica para a proxima execucao")
        continue
    if rodar(["ollama", "pull", modelo], mostrar=False) != 0:
        print(f"[!] falha ao baixar {modelo}; pulando")
        continue
    for fs in FEW_SHOT:
        for particao in PARTICOES:
            if horas() > LIMITE_HORAS:
                print(f"[!] {horas():.1f} h de sessao: {modelo} few-shot {fs} em {particao} fica de fora")
                continue
            cmd = [sys.executable, "src/eval/llm_ordem.py", "--modelo", modelo, "--few-shot", str(fs),
                   "--particao", particao, "--max-blocos", str(MAX_BLOCOS), "--retomar"]
            if modelo.startswith(RACIOCINIO):
                cmd.append("--sem-pensar")
            rodar(cmd)
    if APAGAR_MODELO_APOS_USO:
        rodar(["ollama", "rm", modelo], mostrar=False)
    zipar()

In [ ]:
# Tabela completa: metodos locais (LayoutLM, XY-cut...) + as LLMs que acabaram de rodar, nas mesmas paginas
rodar([sys.executable, "src/eval/comparar.py"])
if os.path.exists("data/golden/_avaliacao/golden_v3/referencia.jsonl"):
    rodar([sys.executable, "src/eval/avaliar_golden.py", "--lote", "_avaliacao/golden_v3", "--layoutlm", "--sucessor"])

In [ ]:
zipar()